## Klasifikasi Topik Artikel Komdigi dengan KNN dan Mutual Information

Artikel dari `title`, `excerpt`, dan `body_text` direpresentasikan dengan Bag-of-Words, lalu `topics` diprediksi menggunakan KNN. Mutual Information memilih token yang paling terkait dengan kelas. Jumlah token dan nilai `k` dipilih pada data validasi berdasarkan Macro F1; data uji final hanya digunakan untuk evaluasi akhir. `topics` adalah kategori artikel, bukan label benar atau salahnya klaim.


In [ ]:
# Import library
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import vstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Cari dataset dari direktori kerja notebook hingga root proyek.
data_path = next(
    (
        parent / "data" / "komdigi_hoaks.csv"
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "data" / "komdigi_hoaks.csv").is_file()
    ),
    None,
)
if data_path is None:
    raise FileNotFoundError("File data/komdigi_hoaks.csv tidak ditemukan dari direktori kerja saat ini.")

df = pd.read_csv(data_path)
required_columns = ["url", "title", "excerpt", "body_text", "topics"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan pada dataset: {missing_columns}")

df = df.drop_duplicates(subset="url", keep="first").dropna(subset=["topics"]).copy()
text_columns = ["title", "excerpt", "body_text"]
df[text_columns] = df[text_columns].fillna("").astype(str)
df["article_text"] = (
    df[text_columns]
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df = df.loc[df["article_text"].ne("")].reset_index(drop=True)

print(f"Lokasi dataset: {data_path.resolve()}")
print("Ukuran dataset setelah deduplikasi:", df.shape)
display(df[["topics", "title", "excerpt"]].head())

In [ ]:
print("Jumlah artikel per topik:")
display(df["topics"].value_counts().rename_axis("topics").to_frame("jumlah"))
print("\nNilai kosong pada teks dan label:")
print(df[["article_text", "topics"]].isnull().sum())

df.info()

plt.figure(figsize=(10, 5))
class_order = df["topics"].value_counts().index
sns.countplot(data=df, y="topics", order=class_order, color="steelblue")
plt.title("Distribusi Topik Artikel Komdigi")
plt.xlabel("Jumlah artikel")
plt.ylabel("Topik")
plt.tight_layout()
plt.show()

In [ ]:
# Fitur teks berasal dari artikel, target adalah kategori topics.
X = df["article_text"]
y = df["topics"]
labels = sorted(y.unique())
print("Jumlah kelas:", len(labels))
print("Kelas:", labels)

# Bagi 20% untuk data uji final, lalu 25% dari sisa data untuk validasi.
# Pemilihan fitur dan hyperparameter tidak melihat label data uji final.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val,
)
print("Data training:", len(X_train))
print("Data validasi:", len(X_val))
print("Data testing final:", len(X_test))

In [ ]:
# Bag-of-Words merepresentasikan kemunculan token; fit vocabulary hanya pada data latih.
bow = CountVectorizer(
    max_features=20000,
    min_df=2,
    ngram_range=(1, 2),
    binary=True,
)
X_train_bow = bow.fit_transform(X_train)
X_val_bow = bow.transform(X_val)
X_test_bow = bow.transform(X_test)

print("Jumlah token:", X_train_bow.shape[1])
print("Dimensi train:", X_train_bow.shape)
print("Dimensi validasi:", X_val_bow.shape)
print("Dimensi test:", X_test_bow.shape)

In [ ]:
# Mutual Information mengukur keterkaitan token dengan kelas topics.
# Token dan label diskret; seleksi dihitung hanya dari data training.
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
mi_scores = mutual_info_classif(
    X_train_bow,
    y_train_encoded,
    discrete_features=True,
    random_state=42,
)

feature_names = bow.get_feature_names_out()
feature_scores = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)
print("Fitur dengan skor Mutual Information tertinggi:")
display(feature_scores.head(20).rename_axis("token").to_frame("mutual_information"))

In [ ]:
# Visualisasikan token teratas berdasarkan Mutual Information.
feature_scores.head(20).sort_values().plot(kind="barh", figsize=(10, 7), color="steelblue")
plt.title("20 Token Teratas Berdasarkan Mutual Information")
plt.xlabel("Skor Mutual Information")
plt.ylabel("Token")
plt.tight_layout()
plt.show()

In [ ]:
# Bandingkan jumlah token teratas dan nilai k menggunakan data validasi.
jumlah_fitur = [50, 100, 250, 500, 1000, 2000]
jumlah_fitur = [n for n in jumlah_fitur if n <= X_train_bow.shape[1]]
nilai_k = [1, 3, 5, 7, 9, 11, 15]
fitur_terurut = feature_scores.index.to_numpy()
hasil = []

for n in jumlah_fitur:
    fitur = fitur_terurut[:n]
    indeks_fitur = bow.vocabulary_
    kolom = np.fromiter((indeks_fitur[token] for token in fitur), dtype=int)
    train_selected = X_train_bow[:, kolom]
    val_selected = X_val_bow[:, kolom]

    for k in nilai_k:
        model = KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine",
            algorithm="brute",
            weights="distance",
        )
        model.fit(train_selected, y_train)
        prediksi = model.predict(val_selected)
        hasil.append({
            "jumlah_fitur": n,
            "k": k,
            "akurasi": accuracy_score(y_val, prediksi),
            "macro_f1": f1_score(y_val, prediksi, average="macro", zero_division=0),
        })

hasil_df = pd.DataFrame(hasil)
display(hasil_df.sort_values(["macro_f1", "akurasi"], ascending=False).head(10))

In [ ]:
# Bandingkan Macro F1 untuk setiap kombinasi jumlah fitur dan nilai k.
plt.figure(figsize=(10, 6))
for n in jumlah_fitur:
    subset = hasil_df[hasil_df["jumlah_fitur"] == n]
    plt.plot(subset["k"], subset["macro_f1"], marker="o", label=f"{n} token")

plt.title("Pemilihan Jumlah Token dan k KNN (Data Validasi)")
plt.xlabel("Nilai k")
plt.ylabel("Macro F1 validasi")
plt.xticks(nilai_k)
plt.legend(title="Fitur")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Pilih konfigurasi terbaik berdasarkan Macro F1, lalu akurasi sebagai tie-break.
hasil_terbaik = hasil_df.sort_values(
    ["macro_f1", "akurasi"], ascending=False
).iloc[0]
jumlah_fitur_terbaik = int(hasil_terbaik["jumlah_fitur"])
nilai_k_terbaik = int(hasil_terbaik["k"])
print("Jumlah token terbaik:", jumlah_fitur_terbaik)
print("Nilai k terbaik:", nilai_k_terbaik)
print("Macro F1 validasi:", round(hasil_terbaik["macro_f1"], 4))
print("Akurasi validasi:", round(hasil_terbaik["akurasi"], 4))

In [ ]:
# Latih ulang model pilihan pada seluruh data train+validasi dan evaluasi sekali pada test.
fitur_terbaik = fitur_terurut[:jumlah_fitur_terbaik]
kolom_terbaik = np.fromiter(
    (bow.vocabulary_[token] for token in fitur_terbaik),
    dtype=int,
)
X_train_val_bow = vstack([X_train_bow, X_val_bow], format="csr")
X_train_val_selected = X_train_val_bow[:, kolom_terbaik]
X_test_selected = X_test_bow[:, kolom_terbaik]
y_train_val = pd.concat([y_train, y_val], ignore_index=True)

knn_terbaik = KNeighborsClassifier(
    n_neighbors=nilai_k_terbaik,
    metric="cosine",
    algorithm="brute",
    weights="distance",
)
knn_terbaik.fit(X_train_val_selected, y_train_val)
y_pred = knn_terbaik.predict(X_test_selected)

print("Akurasi test:", round(accuracy_score(y_test, y_pred), 4))
print("Macro F1 test:", round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4))
print("\nLaporan klasifikasi test:")
print(classification_report(y_test, y_pred, labels=labels, target_names=labels, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=labels,
    display_labels=labels,
    cmap="Blues",
    xticks_rotation=35,
    values_format="d",
)
plt.title("Confusion Matrix KNN - Topik Komdigi")
plt.tight_layout()
plt.show()

In [ ]:
# Lihat hasil prediksi untuk beberapa artikel pada data uji.
hasil_prediksi = df.loc[X_test.index, ["title", "topics"]].rename(
    columns={"topics": "topics_asli"}
).copy()
hasil_prediksi["topics_prediksi"] = y_pred
display(hasil_prediksi.head(10))

In [ ]:
# Visualisasikan hasil validasi untuk jumlah token yang dipilih.
hasil_pilihan = hasil_df[hasil_df["jumlah_fitur"] == jumlah_fitur_terbaik]
plt.figure(figsize=(9, 5))
plt.plot(hasil_pilihan["k"], hasil_pilihan["macro_f1"], marker="o")
plt.axvline(nilai_k_terbaik, color="red", linestyle="--", label=f"k terbaik = {nilai_k_terbaik}")
plt.title(f"Macro F1 Validasi dengan {jumlah_fitur_terbaik} Token Teratas")
plt.xlabel("Nilai k")
plt.ylabel("Macro F1 validasi")
plt.xticks(nilai_k)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()